In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(VGAM)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(piano)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(foreach)
    library(monocle3)
    library(hooke)
    library(ggpubr)
    library(ggrepel)
})

## Load data

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_umap2_final.RDS")

In [ ]:
cds_VCS

In [ ]:
plot_data = as.data.frame(colData(cds_VCS)) %>%
                mutate(
                trajectory_refined = case_when(
                  trajectory_refined == "Brain_and_spinal_cord" ~ "Neuroectoderm_and_glia",
                  trajectory_refined == "Intermediate_progenitors" ~ "Intermediate_neuronal_progenitors",
                  TRUE ~ as.character(trajectory_refined)))

In [ ]:
label_data <- plot_data %>%
                group_by(trajectory_refined) %>%
                summarize(UMAP_2d_1 = median(UMAP_2d_1, na.rm = TRUE),
                          UMAP_2d_2 = median(UMAP_2d_2, na.rm = TRUE)) 


options(repr.plot.width=6, repr.plot.height=6, repr.plot.res=300)
ggplot(plot_data, aes(x = UMAP_2d_1, y = UMAP_2d_2)) +
    geom_point(aes(color = trajectory_refined), size = 0.1, stroke = 0.05, alpha = 0.8) + 

    geom_text_repel(data = label_data, 
                aes(label = str_wrap(str_replace_all(trajectory_refined, "_", " "), width = 10)),
                    size = 3,    
                    color = "black",
                    bg.color = "white", 
                    bg.r = 0.15,
                    segment.color = "black", 
                    segment.size = 0.2,
                    box.padding = 0.7,
                    max.overlaps = Inf  
                ) +
    scale_color_manual(values = my_colors) + 
    labs(x = "UMAP 1", y = "UMAP 2") + 
    monocle3:::monocle_theme_opts() +

    theme(
        axis.title = element_text(color="black", size = 9), 
        axis.text.x = element_text(color="black", size = 9), 
        axis.text.y = element_text(color = "black", size = 9),
        panel.grid.major.x = element_blank(), 
        axis.ticks = element_line(color = "black"),
        legend.position = "none")

In [ ]:
cds_VCS_nuc_area = cds_VCS[, cds_VCS$Marker=="Nucleolin" & cds_VCS$Phenotype == "Area"]
cds_VCS_nuc_ratio = cds_VCS[, cds_VCS$Marker=="Nucleolin" & cds_VCS$Phenotype == "Ratio"]
cds_VCS_sc35_area = cds_VCS[, cds_VCS$Marker=="SC35" & cds_VCS$Phenotype == "Area"]
cds_VCS_sc35_ratio = cds_VCS[, cds_VCS$Marker=="SC35" & cds_VCS$Phenotype == "Ratio"]

In [ ]:
rowdata = as.data.frame(rowData(cds_VCS))

## Gene plotting

## Nucleolar genes

In [ ]:
aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
  cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
  aggregate_marker_expression = counts(cds_gene_list_norm)
  aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
  aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
  aggregate_marker_expression = log10(aggregate_marker_expression + 1)
  colData(cds)[[name]] = aggregate_marker_expression
  return(cds)
}

In [ ]:
cds_VCS_nuc_area = aggregate_gene_score(cds_VCS_nuc_area, c("Ncl"), "Ncl")
cds_VCS_nuc_area = aggregate_gene_score(cds_VCS_nuc_area, c("Npm1"), "Npm1")
cds_VCS_nuc_area = aggregate_gene_score(cds_VCS_nuc_area, c("Fbl"), "Fbl")

cds_VCS_nuc_ratio = aggregate_gene_score(cds_VCS_nuc_ratio, c("Ncl"), "Ncl")
cds_VCS_nuc_ratio = aggregate_gene_score(cds_VCS_nuc_ratio, c("Npm1"), "Npm1")
cds_VCS_nuc_ratio = aggregate_gene_score(cds_VCS_nuc_ratio, c("Fbl"), "Fbl")

In [ ]:
metadata_area = as.data.frame(colData(cds_VCS_nuc_area))
metadata_frac = as.data.frame(colData(cds_VCS_nuc_ratio))

In [ ]:
bin_comparisons <- list( c("Q1", "Q2"), c("Q1", "Q3"), c("Q1", "Q4"),
                         c("Q2", "Q3"), c("Q2", "Q4"), c("Q3", "Q4"))

In [ ]:
## Figure 2D
options(repr.plot.width=6, repr.plot.height=3, repr.plot.res=300)

g = list()

g[[1]] = metadata_frac %>%
    filter(Ncl > 0) %>%
    ggplot(., aes(x = Bin, y = Ncl)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() + 
        stat_compare_means(
            comparisons = bin_comparisons, 
            method = "wilcox.test",
            label = "p.signif" 
        ) +
        #ylim(c(0.04, 0.7)) + 
        labs(y = "Ncl expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black"), 
              axis.text.y = element_text(color = "black"),
              axis.ticks = element_line(color = "black"))

g[[2]] = metadata_frac %>%
    filter(Npm1 > 0) %>%
    ggplot(., aes(x = Bin, y = Npm1)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() +
        stat_compare_means(
            comparisons = bin_comparisons, 
            method = "wilcox.test", 
            label = "p.signif" 
        ) +
        #ylim(c(NA, 0.65)) + 
        labs(y = "Npm1 expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black"), 
              axis.text.y = element_text(color = "black"),
              axis.ticks = element_line(color = "black"))

g[[3]] = metadata_frac %>%
    filter(Fbl > 0) %>%
    ggplot(., aes(x = Bin, y = Fbl)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() + 
        stat_compare_means(
            comparisons = bin_comparisons, 
            method = "wilcox.test",  
            label = "p.signif"   
        ) +
        #ylim(c(NA, 0.6)) + 
        labs(y = "Fbl expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black"), 
              axis.text.y = element_text(color = "black"),
              axis.ticks = element_line(color = "black"))

do.call("grid.arrange", c(g, ncol=3))

In [ ]:
options(repr.plot.width=2.25, repr.plot.height=3, repr.plot.res=300)

## Figure 2D
g = list()

metadata_frac %>%
    filter(Ncl > 0) %>%
    ggplot(., aes(x = Bin, y = Ncl)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.75, fatten = 1) +  
        scale_fill_viridis_d() +  
        ylim(c(0.04, 0.7)) + 
        labs(y = "", x = "") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color = "black", size = 13), 
              axis.text.y = element_text(color = "black", size = 13),
              axis.ticks = element_line(color = "black"),
              axis.line = element_line(color = "black"))

metadata_frac %>%
    filter(Npm1 > 0) %>%
    ggplot(., aes(x = Bin, y = Npm1)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.75, fatten = 1) +  
        scale_fill_viridis_d() + 
        ylim(c(0.04, 0.7)) + 
        labs(y = "", x = "") + # labs(y = "Npm1 expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black", size = 13), 
              axis.text.y = element_text(color = "black", size = 13),
              axis.ticks = element_line(color = "black"),
              axis.line = element_line(color = "black"))

metadata_frac %>%
    filter(Fbl > 0) %>%
    ggplot(., aes(x = Bin, y = Fbl)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.75, fatten = 1) +  
        scale_fill_viridis_d() +    
        ylim(c(0.04, 0.7)) + 
        labs(y = "", x = "") + #labs(y = "Fbl expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black", size = 8), 
              axis.text.y = element_text(color = "black", size = 8),
              axis.ticks = element_line(color = "black"),
              axis.line = element_line(color = "black"))

## Speckle genes

In [ ]:
c("Srsf2", "Son", "Srrm2")

In [ ]:
cds_VCS_sc35_area = aggregate_gene_score(cds_VCS_sc35_area, c("Srsf2"), "Srsf2")
cds_VCS_sc35_area = aggregate_gene_score(cds_VCS_sc35_area, c("Son"), "Son")
cds_VCS_sc35_area = aggregate_gene_score(cds_VCS_sc35_area, c("Srrm2"), "Srrm2")

cds_VCS_sc35_ratio = aggregate_gene_score(cds_VCS_sc35_ratio, c("Srsf2"), "Srsf2")
cds_VCS_sc35_ratio = aggregate_gene_score(cds_VCS_sc35_ratio, c("Son"), "Son")
cds_VCS_sc35_ratio = aggregate_gene_score(cds_VCS_sc35_ratio, c("Srrm2"), "Srrm2")

In [ ]:
metadata_area = as.data.frame(colData(cds_VCS_sc35_area))
metadata_frac = as.data.frame(colData(cds_VCS_sc35_ratio))

In [ ]:
options(repr.plot.width=6, repr.plot.height=3, repr.plot.res=300)

## Supplementary Figure 7C
g = list()

g[[1]] = metadata_frac %>%
    filter(Srsf2 > 0) %>%
    ggplot(., aes(x = Bin, y = Srsf2)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() +  
        stat_compare_means(
            comparisons = bin_comparisons, 
            method = "wilcox.test",  
            label = "p.signif"  
        ) +
        #ylim(c(NA, 0.7)) + 
        labs(y = "Srsf2 expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black"), 
              axis.text.y = element_text(color = "black"),
              axis.ticks = element_line(color = "black"))

g[[2]] = metadata_frac %>%
    filter(Son > 0) %>%
    ggplot(., aes(x = Bin, y = Son)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() +  
        stat_compare_means(
            comparisons = bin_comparisons, 
            method = "wilcox.test",  
            label = "p.signif"  
        ) +
        #ylim(c(NA, 0.7)) + 
        labs(y = "Son expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black"), 
              axis.text.y = element_text(color = "black"),
              axis.ticks = element_line(color = "black"))

g[[3]] = metadata_frac %>%
    filter(Srrm2 > 0) %>%
    ggplot(., aes(x = Bin, y = Srrm2)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() +   
        stat_compare_means(
            comparisons = bin_comparisons, 
            method = "wilcox.test", 
            label = "p.signif"  
        ) +
        #ylim(c(NA, 0.8)) + 
        labs(y = "Srrm2 expression") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black"), 
              axis.text.y = element_text(color = "black"),
              axis.ticks = element_line(color = "black"))

do.call("grid.arrange", c(g, ncol=3))

In [ ]:
speckle_genes_sub = c("Srsf1", "Srsf2", "Srrm2", "Son")
speckle_IDs = as.character(rowdata[rowdata$gene_short_name %in% speckle_genes_sub, ]$gene_ID)

In [ ]:
cds_VCS_sc35_ratio_sub = cds_VCS_sc35_ratio[speckle_IDs,]
cds_VCS_sc35_area_sub = cds_VCS_sc35_area[speckle_IDs,]

In [ ]:
options(repr.plot.width=6.5, repr.plot.height=3, repr.plot.res=300)
g_area = plot_percent_cells_positive(cds_VCS_sc35_area_sub, group_cells_by = "Bin", 
                                     nrow = 1, ncol = 4, min_expr = 0.1) + 
            scale_fill_viridis_d() +
            theme(axis.text.x = element_text(color="black", angle = 0, hjust = 0.5, size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"), legend.position = "none")
g_fraction = plot_percent_cells_positive(cds_VCS_sc35_ratio_sub, group_cells_by = "Bin", 
                                         nrow = 1, ncol = 4, min_expr = 0.1) +
            scale_fill_viridis_d() +
            theme(axis.text.x = element_text(color="black", angle = 0, hjust = 0.5, size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"), legend.position = "none")

In [ ]:
## Figure 2E, Supplementary Figure 7D

options(repr.plot.width=6.5, repr.plot.height=3.5, repr.plot.res=300)
g_fraction 
g_area

In [ ]:
options(repr.plot.width=2.25, repr.plot.height=3, repr.plot.res=300)

## Supplementary Figure 7C
g = list()

metadata_frac %>%
    filter(Srsf2 > 0) %>%
    ggplot(., aes(x = Bin, y = Srsf2)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() +   
        ylim(c(0.04, 0.7)) +  
        labs(y = "", x = "") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black", size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"))

metadata_frac %>%
    filter(Son > 0) %>%
    ggplot(., aes(x = Bin, y = Son)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() +   
        ylim(c(0.04, 0.7)) + 
        labs(y = "", x = "") +  
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black", size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"))

metadata_frac %>%
    filter(Srrm2 > 0) %>%
    ggplot(., aes(x = Bin, y = Srrm2)) +
        geom_violin(aes(fill = Bin), trim = FALSE, alpha = 0.9, color = NA) +
        geom_boxplot(fill = "white", width = 0.25, outlier.shape = NA, lwd = 0.5, fatten = 1) +  
        scale_fill_viridis_d() +   
        ylim(c(0.04, 0.8)) + 
        labs(y = "", x = "") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color="black", size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"))

In [ ]:
rowdata = rowData(cds_VCS)

nucleolus_genes = c("Ncl", "Npm1", "Fbl", "Polr1a", "Polr1b",
                    "Nop58", "Nop56", "Gar1", 
                    "Ddx21", "Wdr12", "Rcl1", "Ubtf")

speckle_genes = c("Srsf1", "Srsf2", "Son", "Srrm2", "Srsf11", "Srsf7", "Srsf4", "Srsf5",  "Prpf8")

In [ ]:
nucleolus_ID = as.character(rowdata[rowdata$gene_short_name %in% nucleolus_genes, ]$gene_ID)
speckle_ID = as.character(rowdata[rowdata$gene_short_name %in% speckle_genes, ]$gene_ID)

In [ ]:
cds_VCS_nuc_area$Bin = factor(cds_VCS_nuc_area$Bin, levels = c("Q1", "Q2", "Q3", "Q4"))
cds_VCS_nuc_ratio$Bin = factor(cds_VCS_nuc_ratio$Bin, levels = c("Q1", "Q2", "Q3", "Q4"))

In [ ]:
## Supplementary Figure 7A
options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=200)
plot_genes_by_group(cds_VCS_nuc_area[nucleolus_ID,], markers = nucleolus_genes, max.size = 5, 
                    lower_threshold = 0.1, ordering_type = "none",
                    group_cells_by = "Bin") +
            labs(x = "", y = "") + 
            theme(axis.text.x = element_text(color="black", angle = 0, hjust = 0.5, size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"))

options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=200)
plot_genes_by_group(cds_VCS_nuc_ratio[nucleolus_ID,], markers = nucleolus_genes, max.size = 5, 
                    lower_threshold = 0.1, ordering_type = "none", 
                    group_cells_by = "Bin") +
            labs(x = "", y = "") + 
            theme(axis.text.x = element_text(color="black", angle = 0, hjust = 0.5, size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"))

In [ ]:
## Supplementary Figure 7B

options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=200)
plot_genes_by_group(cds_VCS_sc35_area[speckle_ID,], markers = speckle_genes, max.size = 4, ordering_type = "none",
                    lower_threshold = 0.1, group_cells_by = "Bin") +
            labs(x = "", y = "") + 
            theme(axis.text.x = element_text(color="black", angle = 0, hjust = 0.5, size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"))

options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=200)
plot_genes_by_group(cds_VCS_sc35_ratio[speckle_ID,], markers = speckle_genes, max.size = 4, ordering_type = "none",
                    lower_threshold = 0.1, group_cells_by = "Bin") +
            labs(x = "", y = "") + 
            theme(axis.text.x = element_text(color="black", angle = 0, hjust = 0.5, size=13), 
              axis.text.y = element_text(color = "black", size=13),
              axis.ticks = element_line(color = "black"))

## By cell type

In [ ]:
subsample_cds = function(cds) {
    metadata = as.data.frame(colData(cds))
    count_df = metadata %>% count(Experiment, Embryo, Marker, Phenotype, Bin)


    cells_sampled = c()

    for (i in 1:nrow(count_df)) {
        marker = count_df[i,]$Marker
        phenotype = count_df[i,]$Phenotype
        bin = count_df[i,]$Bin
        experiment = count_df[i,]$Experiment
        embryo = count_df[i,]$Embryo

        n = min(count_df[count_df$Marker == marker & count_df$Phenotype == phenotype & 
                         count_df$Experiment == experiment & count_df$Embryo == embryo,]$n)

        cells = metadata %>% filter(Marker == marker & Phenotype == phenotype & Bin == bin & 
                                    Experiment == experiment & Embryo == embryo) %>% pull(Cell)
        cells_sampled = c(cells_sampled, sample(cells, size=n, replace=F))
    }

    cds_subsampled = cds[, cds$Cell %in% cells_sampled]
    
    return(cds_subsampled)
}

compute_bin_score = function(cds, downsample = TRUE, n_min = 30) {
    if (downsample) {
        cds_subsampled = subsample_cds(cds)
    } else {
        cds_subsampled = cds
    }
    
    metadata = as.data.frame(colData(cds_subsampled))

    bin_score_list_cluster = list()

    for (marker in unique(metadata$Marker)) {
        bin_score_list_cluster[[marker]] = list()
        for (phenotype in unique(metadata[metadata$Marker == marker,]$Phenotype)) {

            bin_scores = metadata %>%
                filter(Marker == marker & Phenotype == phenotype) %>%
                group_by(Bin, nn_majority_cell_type_pca) %>%
                count() %>%
                as.data.frame() %>%
                tidyr::spread(Bin, n, fill = 0) %>%
                #rowwise() %>%
                mutate(n = Q1+Q2+Q3+Q4) %>%
                as.data.frame() %>%
                filter(n >= n_min)

            bin_scores = bin_scores %>%
                            mutate(nn_majority_cell_type_pca = factor(nn_majority_cell_type_pca, 
                                                                levels = unique(bin_scores$nn_majority_cell_type_pca)),
                                    Marker = marker,
                                    Phenotype = phenotype)

            bin_score_list_cluster[[marker]][[phenotype]] = bin_scores
        }
    }
    
    bin_score_concat = NULL
    for (marker in names(bin_score_list_cluster)) {
        df = do.call("rbind", bin_score_list_cluster[[marker]])
        if (is.null(bin_score_concat)) {
            bin_score_concat = df
        } else {
            bin_score_concat = rbind(bin_score_concat, df)
        }
    }

    return(bin_score_concat)
    
}

In [ ]:
bin_score_concat = compute_bin_score(cds_VCS, F, 50)

In [ ]:
bin_score_concat = bin_score_concat %>%
    mutate(Q1 = Q1 / n, Q2 = Q2 / n, Q3 = Q3 / n, Q4 = Q4 / n,
           Phenotype = case_match(Phenotype, "Ratio" ~ "Fraction", .default = Phenotype),
           Phenotypes = paste0(Marker, '-', Phenotype)) %>%
    filter(n > 200) 

In [ ]:
bin_score_all = NULL

for (phenotype in c("Nucleolin-Fraction")) {
    bin_score_sub = bin_score_concat %>%
                        filter(Phenotypes == phenotype) %>%
                        select(-c(n, Marker, Phenotype, Phenotypes))
    
    if (is.null(bin_score_all)) {
        bin_score_all = bin_score_sub
    } else {
        bin_score_all = bin_score_all %>%
                            left_join(., bin_score_sub, by = "nn_majority_cell_type_pca")
    }
} 

    
bin_scores_df = bin_score_all %>%
                    na.omit() 

bin_score_all = bin_score_all %>% 
                    na.omit() %>% 
                    `row.names<-`(., NULL) %>%
                    column_to_rownames("nn_majority_cell_type_pca")


head(bin_score_all)
head(bin_scores_df)

In [ ]:
## Supplementary Figure 7H

options(repr.plot.width=18, repr.plot.height=3.8, repr.plot.res=400)
ph = pheatmap(t(bin_score_all), cluster_cols = T, angle_col = 45, cluster_rows = F, silent = T)

ph$gtable$widths[[1]] <- unit(1.2, "inches")

grid.newpage()
grid.draw(ph$gtable)

## By main lineage

In [ ]:
subsample_cds = function(cds) {
    metadata = as.data.frame(colData(cds))
    count_df = metadata %>% count(Experiment, Embryo, Marker, Phenotype, Bin)


    cells_sampled = c()

    for (i in 1:nrow(count_df)) {
        marker = count_df[i,]$Marker
        phenotype = count_df[i,]$Phenotype
        bin = count_df[i,]$Bin
        experiment = count_df[i,]$Experiment
        embryo = count_df[i,]$Embryo

        n = min(count_df[count_df$Marker == marker & count_df$Phenotype == phenotype & 
                         count_df$Experiment == experiment & count_df$Embryo == embryo,]$n)

        cells = metadata %>% filter(Marker == marker & Phenotype == phenotype & Bin == bin & 
                                    Experiment == experiment & Embryo == embryo) %>% pull(Cell)
        cells_sampled = c(cells_sampled, sample(cells, size=n, replace=F))
    }

    cds_subsampled = cds[, cds$Cell %in% cells_sampled]
    
    return(cds_subsampled)
}

compute_bin_score = function(cds, downsample = TRUE, n_min = 30) {
    if (downsample) {
        cds_subsampled = subsample_cds(cds)
    } else {
        cds_subsampled = cds
    }
    
    metadata = as.data.frame(colData(cds_subsampled))

    bin_score_list_cluster = list()

    for (marker in unique(metadata$Marker)) {
        bin_score_list_cluster[[marker]] = list()
        for (phenotype in unique(metadata[metadata$Marker == marker,]$Phenotype)) {

            bin_scores = metadata %>%
                filter(Marker == marker & Phenotype == phenotype) %>%
                group_by(Bin, trajectory_refined) %>%
                count() %>%
                as.data.frame() %>%
                tidyr::spread(Bin, n, fill = 0) %>%
                rowwise() %>%
                mutate(n = Q1+Q2+Q3+Q4) %>%
                as.data.frame() %>%
                filter(n >= n_min) 

            bin_scores = bin_scores %>%
                            mutate(trajectory_refined = factor(trajectory_refined, 
                                                              levels = unique(bin_scores$trajectory_refined)),
                                    Marker = marker,
                                    Phenotype = phenotype)

            bin_score_list_cluster[[marker]][[phenotype]] = bin_scores
        }
    }
    
    bin_score_concat = NULL
    for (marker in names(bin_score_list_cluster)) {
        df = do.call("rbind", bin_score_list_cluster[[marker]])
        if (is.null(bin_score_concat)) {
            bin_score_concat = df
        } else {
            bin_score_concat = rbind(bin_score_concat, df)
        }
    }

    return(bin_score_concat)
    
}

In [ ]:
bin_score_concat = compute_bin_score(cds_VCS, F, 50) %>%
    mutate(Q1 = Q1 / n, Q2 = Q2 / n, Q3 = Q3 / n, Q4 = Q4 / n,
           Phenotype = case_match(Phenotype, "Ratio" ~ "Fraction", .default = Phenotype),
           Phenotypes = paste0(Marker, '-', Phenotype))
bin_score_concat

In [ ]:
phenotype = "Nucleolin-Fraction"

bin_score_nuc_frac = bin_score_concat %>%
                        filter(Phenotypes == phenotype & n >= 100) %>%
                        select(-c(n, Marker, Phenotype, Phenotypes)) %>%
                        na.omit() %>% 
                        `row.names<-`(., NULL) %>%
                        column_to_rownames("trajectory_refined")

head(bin_score_nuc_frac)

In [ ]:
dim(bin_score_nuc_frac)

In [ ]:
bin_score_nuc_frac %>%
    filter(Q4 > 0.5)

In [ ]:
row_labels = rownames(bin_score_nuc_frac)
row_labels[3] = "Neuroectoderm_and_glia"
row_labels[11] = "Intermediate_neuronal_progenitors"

row_labels = gsub("_", " ", row_labels)

rownames(bin_score_nuc_frac) = row_labels

In [ ]:
gg = pheatmap(bin_score_nuc_frac, cluster_cols = F, cluster_rows = T, annotation_names_row = F,
              annotation_colors = mycolors, silent = TRUE) 

In [ ]:
## Figure 2G
options(repr.plot.width=5.5, repr.plot.height=4, repr.plot.res=300)

cols <- colorRampPalette(RColorBrewer::brewer.pal(8, "Dark2"))
n = 5
row_clusters = cutree(gg$tree_row, n)

mycolors = cols(n)
names(mycolors) <- 1:n
mycolors <- list(Cluster = mycolors)
annotdf = cutree(gg$tree_row, 5)

gg = pheatmap(bin_score_nuc_frac, cluster_cols = F, cluster_rows = T, annotation_names_row = F,
              annotation_row = data.frame(Cluster = as.factor(annotdf)),
              annotation_colors = mycolors, angle_col = 0) 

In [ ]:
phenotype = "SC35-Fraction"

bin_score_sc35_frac = bin_score_concat %>%
                        filter(Phenotypes == phenotype) %>%
                        select(-c(n, Marker, Phenotype, Phenotypes)) %>%
                        na.omit() %>% 
                        `row.names<-`(., NULL) %>%
                        column_to_rownames("trajectory_refined")

head(bin_score_sc35_frac)

In [ ]:
options(repr.plot.width=4, repr.plot.height=3.5, repr.plot.res=300)
gg = pheatmap(t(bin_score_sc35_frac), cluster_cols = T, cluster_rows = F,
             silent=T) 

In [ ]:
row_labels = rownames(bin_score_sc35_frac)
row_labels[3] = "Neuroectoderm_and_glia"
row_labels[11] = "Intermediate_neuronal_progenitors"

row_labels = gsub("_", " ", row_labels)

rownames(bin_score_sc35_frac) = row_labels

In [ ]:
## Supplementary Figure 7G
options(repr.plot.width=4.5, repr.plot.height=4, repr.plot.res=300)
gg = pheatmap(bin_score_sc35_frac, cluster_cols = F, cluster_rows = T, annotation_names_row = F,
             angle_col = 0) 

In [ ]:
phenotype = "Nucleolin-Area"

bin_score_nuc_area = bin_score_concat %>%
                        filter(Phenotypes == phenotype) %>%
                        select(-c(n, Marker, Phenotype, Phenotypes)) %>%
                        na.omit() %>% 
                        `row.names<-`(., NULL) %>%
                        column_to_rownames("trajectory_refined")

head(bin_score_nuc_area)

In [ ]:
## Supplementary Figure 7E

row_labels = rownames(bin_score_nuc_area)
row_labels[3] = "Neuroectoderm_and_glia"
row_labels[11] = "Intermediate_neuronal_progenitors"

row_labels = gsub("_", " ", row_labels)

rownames(bin_score_nuc_area) = row_labels

options(repr.plot.width=4.5, repr.plot.height=4, repr.plot.res=300)

gg = pheatmap(bin_score_nuc_area, cluster_cols = F, cluster_rows = T, annotation_names_row = F,
             angle_col = 0) 

In [ ]:
phenotype = "SC35-Area"

bin_score_sc35_area = bin_score_concat %>%
                        filter(Phenotypes == phenotype) %>%
                        select(-c(n, Marker, Phenotype, Phenotypes)) %>%
                        na.omit() %>% 
                        `row.names<-`(., NULL) %>%
                        column_to_rownames("trajectory_refined")

head(bin_score_sc35_area)

In [ ]:
row_labels = rownames(bin_score_sc35_area)
row_labels[3] = "Neuroectoderm_and_glia"
row_labels[11] = "Intermediate_neuronal_progenitors"

row_labels = gsub("_", " ", row_labels)

rownames(bin_score_sc35_area) = row_labels

options(repr.plot.width=4.5, repr.plot.height=4, repr.plot.res=300)

gg = pheatmap(bin_score_sc35_area, cluster_cols = F, cluster_rows = T, annotation_names_row = F,
             angle_col = 0) 